# Data Engineer Testing

Human-run notebook for the Step 4 Data Engineer slice.

Run the cells top to bottom to:
- resolve the repo root even if Jupyter starts in `notebooks/`
- run the focused Data Engineer tests
- run the broader Step 4 validation set


In [11]:
from pathlib import Path
import subprocess

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from the current notebook working directory.")

from pprint import pprint
from tempfile import TemporaryDirectory

import pandas as pd

from multi_agent_ds.skills.cleaning import apply_cleaning_actions
from multi_agent_ds.skills.feature_engineering import apply_feature_actions
from multi_agent_ds.workflows.preparation import run_preparation_workflow
from multi_agent_ds.agents.data_engineer import data_engineer_node

ROOT = resolve_repo_root()
print("Repo root:", ROOT)
print("Notebook cwd:", Path.cwd().resolve())

def run_pytest(args: list[str]) -> None:
    cmd = ["uv", "run", "pytest", *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=ROOT)
    if completed.returncode != 0:
        raise RuntimeError(f"pytest failed with exit code {completed.returncode}")


Repo root: /Users/sean.lewis/DataspellProjects/multi_agent_ds
Notebook cwd: /Users/sean.lewis/DataspellProjects/multi_agent_ds/notebooks


## Focused Data Engineer Agent Tests

Use this first when checking the current agent/output contract changes.

In [12]:
run_pytest([
    "tests/test_pre_modeling_review_agents.py",
])


Running: uv run pytest tests/test_pre_modeling_review_agents.py
============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/sean.lewis/DataspellProjects/multi_agent_ds
configfile: pyproject.toml
plugins: cov-7.1.0, Faker-40.13.0, langsmith-0.7.30, anyio-4.13.0
collected 4 items

tests/test_pre_modeling_review_agents.py ....                            [100%]

=============================== warnings summary ===============================
.venv/lib/python3.11/site-packages/rdt/transformers/utils.py:18
  /Users/sean.lewis/DataspellProjects/multi_agent_ds/.venv/lib/python3.11/site-packages/rdt/transformers/utils.py:18: DeprecationWarning: module 'sre_parse' is deprecated
    import sre_parse  # isort:skip

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
========================= 4 passed, 1 warning in 2.63s =========================


## Focused Step 4 Validation

This runs the Data Engineer skill and workflow checks together.

In [13]:
run_pytest([
    "tests/test_cleaning.py",
    "tests/test_feature_engineering.py",
    "tests/test_preparation_workflow.py",
    "tests/test_pre_modeling_review_agents.py",
])


Running: uv run pytest tests/test_cleaning.py tests/test_feature_engineering.py tests/test_preparation_workflow.py tests/test_pre_modeling_review_agents.py
============================= test session starts ==============================
platform darwin -- Python 3.11.14, pytest-9.0.3, pluggy-1.6.0
rootdir: /Users/sean.lewis/DataspellProjects/multi_agent_ds
configfile: pyproject.toml
plugins: cov-7.1.0, Faker-40.13.0, langsmith-0.7.30, anyio-4.13.0
collected 14 items

tests/test_cleaning.py ....                                              [ 28%]
tests/test_feature_engineering.py ...                                    [ 50%]
tests/test_preparation_workflow.py ...                                   [ 71%]
tests/test_pre_modeling_review_agents.py ....                            [100%]

=============================== warnings summary ===============================
.venv/lib/python3.11/site-packages/rdt/transformers/utils.py:18
  /Users/sean.lewis/DataspellProjects/multi_agent_ds/.venv/lib

## Example Passing Outputs

These cells print the actual structured outputs that the Step 4 code returns on a small fixture dataset.

In [14]:
fixture_df = pd.DataFrame(
    {
        "claim_amount_avg": [10.0, 12.0, 1000.0],
        "distance_to_work": [1.0, None, 4.0],
        "annual_premium": [2.0, 0.0, 5.0],
        "segment": ["a", None, "b"],
        "binary_target": [0, 1, 1],
    }
)
fixture_df


,claim_amount_avg,distance_to_work,annual_premium,segment,binary_target
0,10.0,1.0,2.0,a,0
1,12.0,NaN,0.0,None,1
2,1000.0,4.0,5.0,b,1


In [15]:
cleaned_df, cleaning_summary = apply_cleaning_actions(
    fixture_df,
    actions=[
        {"action": "clip_outliers_iqr", "params": {"columns": ["claim_amount_avg"]}},
        {"action": "impute_numeric_median", "params": {"columns": ["distance_to_work"]}},
        {"action": "impute_categorical_mode", "params": {"columns": ["segment"]}},
    ],
    target_col="binary_target",
)
print("Cleaning summary:")
pprint(cleaning_summary)
print("\nCleaned dataframe:")
cleaned_df


Cleaning summary:
[{'action': 'clip_outliers_iqr',
  'applied_columns': [{'clipped_count': 0,
                       'column': 'claim_amount_avg',
                       'lower_bound': -731.5,
                       'upper_bound': 1248.5}],
  'multiplier': 1.5,
  'requested_columns': ['claim_amount_avg'],
  'skipped_columns': [],
  'status': 'applied'},
 {'action': 'impute_numeric_median',
  'applied_columns': [{'column': 'distance_to_work',
                       'fill_value': 2.5,
                       'filled_count': 1}],
  'requested_columns': ['distance_to_work'],
  'skipped_columns': [],
  'status': 'applied'},
 {'action': 'impute_categorical_mode',
  'applied_columns': [{'column': 'segment',
                       'fill_value': 'a',
                       'filled_count': 1}],
  'requested_columns': ['segment'],
  'skipped_columns': [],
  'status': 'applied'}]

Cleaned dataframe:


,claim_amount_avg,distance_to_work,annual_premium,segment,binary_target
0,10.0,1.0,2.0,a,0
1,12.0,2.5,0.0,a,1
2,1000.0,4.0,5.0,b,1


In [16]:
engineered_df, feature_summary = apply_feature_actions(
    cleaned_df,
    actions=[
        {"action": "log1p", "params": {"columns": ["claim_amount_avg"]}},
        {
            "action": "ratio",
            "params": {
                "numerator": "claim_amount_avg",
                "denominator": "annual_premium",
                "output_column": "claim_per_premium",
            },
        },
    ],
    target_col="binary_target",
)
print("Feature summary:")
pprint(feature_summary)
print("\nEngineered dataframe columns:")
print(engineered_df.columns.tolist())
engineered_df


Feature summary:
[{'action': 'log1p',
  'created_features': [{'feature': 'claim_amount_avg_log1p',
                        'source_column': 'claim_amount_avg',
                        'transform': 'log1p_clip_nonnegative'}],
  'requested_columns': ['claim_amount_avg'],
  'skipped': [],
  'status': 'applied'},
 {'action': 'ratio',
  'created_features': [{'denominator': 'annual_premium',
                        'feature': 'claim_per_premium',
                        'numerator': 'claim_amount_avg',
                        'zero_denominator_count': 1}],
  'output_column': 'claim_per_premium',
  'requested_columns': ['claim_amount_avg', 'annual_premium'],
  'skipped': [],
  'status': 'applied'}]

Engineered dataframe columns:
['claim_amount_avg', 'distance_to_work', 'annual_premium', 'segment', 'binary_target', 'claim_amount_avg_log1p', 'claim_per_premium']


,claim_amount_avg,distance_to_work,annual_premium,segment,binary_target,claim_amount_avg_log1p,claim_per_premium
0,10.0,1.0,2.0,a,0,2.397895,5.0
1,12.0,2.5,0.0,a,1,2.564949,<NA>
2,1000.0,4.0,5.0,b,1,6.908755,200.0


In [17]:
with TemporaryDirectory() as tmp_dir:
    source_path = Path(tmp_dir) / "raw.parquet"
    fixture_df.to_parquet(source_path)

    import multi_agent_ds.workflows.preparation as prep_module

    original_upload = prep_module.upload_to_s3

    def fake_upload_to_s3(local_path, path_key, filename, settings):
        print(f"Fake upload: {local_path} -> {path_key}/{filename}")
        return f"s3://{settings['s3']['bucket']}/{settings['s3']['prefix']}/{settings['s3']['paths'][path_key]}/{filename}"

    prep_module.upload_to_s3 = fake_upload_to_s3
    try:
        prep_result = run_preparation_workflow(
            data_path=str(source_path),
            prep_plan={
                "cleaning_actions": [
                    {"action": "clip_outliers_iqr", "params": {"columns": ["claim_amount_avg"]}},
                    {"action": "impute_numeric_median", "params": {"columns": ["distance_to_work"]}},
                ],
                "feature_actions": [
                    {
                        "action": "ratio",
                        "params": {
                            "numerator": "claim_amount_avg",
                            "denominator": "distance_to_work",
                            "output_column": "claim_per_distance",
                        },
                    }
                ],
            },
            settings={
                "data": {"source": "existing", "existing": {"target_column": "binary_target"}},
                "s3": {
                    "bucket": "example-bucket",
                    "prefix": "multi_agent_ds",
                    "paths": {"processed": "data/processed"},
                },
            },
        )
    finally:
        prep_module.upload_to_s3 = original_upload

print("Preparation workflow result:")
pprint(prep_result)


Fake upload: /var/folders/gn/hsgn5mm928xf1m321skf0j880000gp/T/tmp1ckemxkz/raw_processed_20260416T185735Z.parquet -> processed/raw_processed_20260416T185735Z.parquet
Preparation workflow result:
{'artifact_filename': 'raw_processed_20260416T185735Z.parquet',
 'cleaning_summary': [{'action': 'clip_outliers_iqr',
                       'applied_columns': [{'clipped_count': 0,
                                            'column': 'claim_amount_avg',
                                            'lower_bound': -731.5,
                                            'upper_bound': 1248.5}],
                       'multiplier': 1.5,
                       'requested_columns': ['claim_amount_avg'],
                       'skipped_columns': [],
                       'status': 'applied'},
                      {'action': 'impute_numeric_median',
                       'applied_columns': [{'column': 'distance_to_work',
                                            'fill_value': 2.5,
                    

In [18]:
def fake_run_preparation_workflow(data_path, prep_plan, settings):
    return {
        "source_data_path": data_path,
        "target_column": "binary_target",
        "artifact_filename": "example_processed_20260416T010203Z.parquet",
        "processed_data_path": "s3://bucket/prefix/data/processed/example_processed_20260416T010203Z.parquet",
        "source_n_rows": 3,
        "source_n_features": 4,
        "n_rows": 3,
        "n_features": 5,
        "processed_n_rows": 3,
        "processed_n_features": 5,
        "cleaning_summary": [{"action": "clip_outliers_iqr"}],
        "feature_summary": [{"action": "ratio"}],
    }

import multi_agent_ds.agents.data_engineer as data_engineer_module

original_run_preparation_workflow = data_engineer_module.run_preparation_workflow
data_engineer_module.run_preparation_workflow = fake_run_preparation_workflow
try:
    execute_result = data_engineer_node(
        {
            "settings": {"llm": {"providers": {"openai": {"model": "gpt-4o", "temperature": 0.2, "max_tokens": 1000}}}},
            "data_path": "data/raw/example.parquet",
            "prep_plan": {"cleaning_actions": [{"action": "clip_outliers_iqr"}], "feature_actions": []},
            "agent_decisions": [],
        },
        mode="execute",
    )
finally:
    data_engineer_module.run_preparation_workflow = original_run_preparation_workflow

print("data_engineer execute-mode result:")
pprint(execute_result)


data_engineer execute-mode result:
{'agent_decisions': [{'agent': 'data_engineer',
                      'artifact_filename': 'example_processed_20260416T010203Z.parquet',
                      'phase': 'prep_execute',
                      'processed_data_path': 's3://bucket/prefix/data/processed/example_processed_20260416T010203Z.parquet',
                      'processed_n_features': 5,
                      'processed_n_rows': 3,
                      'target_column': 'binary_target'}],
 'current_phase': 'prep_execute',
 'prep_result': {'artifact_filename': 'example_processed_20260416T010203Z.parquet',
                 'cleaning_summary': [{'action': 'clip_outliers_iqr'}],
                 'feature_summary': [{'action': 'ratio'}],
                 'n_features': 5,
                 'n_rows': 3,
                 'processed_data_path': 's3://bucket/prefix/data/processed/example_processed_20260416T010203Z.parquet',
                 'processed_n_features': 5,
                 'processed

## Optional Current Worktree Check

In [19]:
subprocess.run(["git", "status", "--short"], cwd=ROOT, check=True)


 M notebooks/Data_Engineer_Testing.ipynb


CompletedProcess(args=['git', 'status', '--short'], returncode=0)